# Introduction to Hopfield Networks
In this module, we will introduce traditional and modern Hopfield networks, which are a type of recurrent neural network (RNN) that can be used for associative memory tasks. By the end of this module, you will be able to define and demonstrate mastery of the following key concepts:

* __Recurrent Neural Networks (RNNs)__ are a type of artificial neural network designed to process sequential data by retaining information about previous time steps through an _internal memory_. This makes them particularly effective for tasks such as language modeling and time-series prediction where context and dependencies between data points are crucial.

* __Traditional Hopfield networks__ are fully connected, recurrent neural networks composed of binary threshold units that store memory patterns as minima of a global energy function. By asynchronously updating each neuron to reduce the network’s energy, the system converges to the nearest stored pattern, acting as a content-addressable associative memory. 

* __Modern Hopfield networks__ are generalizations of classical Hopfield networks that expand their capabilities to store and retrieve high-dimensional continuous data. They are designed to overcome the limitations of classical Hopfield networks, which are constrained by linear scaling relationships between the number of input features and the number of stored memories, discrete representations, and sensitivity to correlated patterns.

Hopfield networks are a fascinating area of research in the field of neural networks and have applications in various domains, including image recognition, natural language processing, and optimization problems. So let's get started!
___

## Motivation: Modeling Sequences
Suppose we have a _sequence of data_ $(x_1, x_2, \ldots, x_T)$ where $T$ is the sequence length, and $x_i$ is the $i$-th element (token) of the sequence. 
* _Example sequences_: in natural language processing, $x_{i}$ could be words or characters in a sentence in a word. In time series analysis, $x_t$ could be a measurement, i.e., temperature, pressure, price, etc, at time $i$.

To model this sequence, i.e., predict the next token given past tokens, we _could try_ to use tools such as Hidden Markov Models (HMMs). However, HMMs are limited in their ability to capture _long-range dependencies_ and complex relationships between elements in the sequence. 
* __Why?__ HMMs use the _Markov property_, which says that the future state of a system depends only on its current state and not on its past states. This assumption is often too restrictive for many real-world applications, where the relationships between elements in a sequence can be more complex and require a more flexible modeling approach.

This is where recurrent neural networks (RNNs) come in. RNNs are designed to handle data sequences by maintaining a hidden state that captures information about previous inputs. This allows them to model long-range dependencies and contextual relationships between elements in the sequence.

## What are RNNs?
Recurrent Neural Networks (RNNs) are artificial neural networks designed to process sequential data by retaining information about previous inputs through their internal memory. 

* _Do feedforward neural networks have memory?_ No, feedforward neural networks process do not retain information about previous inputs. Thus, the parameters (weights and bias values) do not change once training is over. This means that the network is done learning and evolving. When we feed in values, an FNN applies the operations that make up the network using the values it has learned.
* _How are RNNs different from feedforward neural networks?_ RNNs have connections that loop back on themselves, allowing them to maintain a _hidden state_ that captures information about previous inputs. This makes RNNs particularly effective for tasks such as language modeling, time-series prediction, and speech recognition, where context and dependencies between data points are crucial. 

Let's look at a _simple_ RNNs: the Elman network.

<img
  src="figs/recurrent_neural_network_unfold.svg"
  alt="triangle with all three sides equal"
  height="400"
  width="800" />

### Elman Network: Mathematical Formulation
The Elman network is a simple RNN type consisting of an input layer, a hidden layer, and an output layer. The hidden layer has recurrent connections that allow it to maintain a hidden state over time:

* [Elman, J. L. (1990). Finding structure in time. Cognitive Science, 14(2), 179-211.](https://onlinelibrary.wiley.com/doi/10.1207/s15516709cog1402_1)

__At each time step__: an Elman RNN takes an _input_ and the previous hidden state (memory) and computes the output entry at time $t$. 

Let the input vector at time $t$ be denoted as $\mathbf{x}_t\in\mathbb{R}^{d_{in}}$, the hidden state at time $t$ as $\mathbf{h}_t\in\mathbb{R}^{h}$, and the output at time $t$ as $\mathbf{y}_t\in\mathbb{R}^{d_{out}}$. The following equations can describe the RNN:
$$
\begin{align*}
\mathbf{h}_t &= \sigma_{h}(\mathbf{U}_h \mathbf{h}_{t-1} + \mathbf{W}_x \mathbf{x}_t + \mathbf{b}_h) \\
\mathbf{y}_t &= \sigma_{y}(\mathbf{W}_y \mathbf{h}_t + \mathbf{b}_y)
\end{align*}
$$
where the parameters are:
* _Network weights_: the term $\mathbf{U}_h\in\mathbb{R}^{h\times{h}}$ is the weight matrix for the hidden state, $\mathbf{W}_x\in\mathbb{R}^{h\times{d_{in}}}$ is the weight matrix for the input, and $\mathbf{W}_y\in\mathbb{R}^{d_{out}\times{h}}$ is the weight matrix for the output
* _Network bias_: the $\mathbf{b}_h\in\mathbb{R}^{h}$ terms denote the bias vector for the hidden state, and $\mathbf{b}_y\in\mathbb{R}^{d_{out}}$ is the bias vector for the output.
* _Activation function_: the $\sigma_{h}$ function is a _hidden layer activation function_, such as the sigmoid or hyperbolic tangent (tanh) function, which introduces non-linearity into the RNN. The activation function $\sigma_{y}$ is an _output activation function_ that can be a softmax function for classification tasks or a linear function for regression tasks.

How many parameters are there in the Elman network? The number of parameters in an Elman RNN can be calculated as follows:
* _Hidden state_: The number of parameters for the hidden state is $h^2 + d_{in}h + h = h(h + d_{in} + 1)$
* _Output_: The number of parameters for the output is $d_{out}h + d_{out} = d_{out}(h + 1)$
* _Total_: The total number of parameters in the Elman RNN is $h(h + d_{in} + 1) + d_{out}(h + 1)$

___

## Classical Hopfield Networks
A classical Hopfield network is a fully connected undirected graph consisting of $N$ nodes, where each node in the graph has a state $s = \pm{1}$; each node is connected to every other node, but not to itself, i.e., the network has no self-loops. The weights of the connection between node $i$ and $j$, denoted as $w_{ij}\in\mathbf{W}$ are learned using a __Hebbian learning rule__.
* __Hebbian learning__ proposed by Donald Hebb in 1949 says that synaptic connections between neurons are strengthened when they activate (fire) simultaneously, forming the biological basis for __associative learning__. This "fire together, wire together" principle underpins unsupervised learning in neural networks, linking co-active nodes to enable pattern storage and adaptation.
* __Different?__ Unlike the previous examples of learning, e.g., the Perceptron, the parameters (weights) in a classical Hopfield network are entirely specified by the memories we want to encode. Thus, we do not need to search for weights or learn them by experimenting with the world. Instead, we can directly compute the weights from the memories we want to encode.

### Encoding memories into a Hopfield network
Suppose we wish our network to memorize $K$-images, where each image is an $n\times{n}$ collection of black and white pixels represented as a vector $\mathbf{s}_{i}\in\left\{-1,1\right\}\in{R}^{n^2}$. We encode the image using the following rule: if the pixel is white, we set the memory value to `1`, and if the pixel is black, we set the memory value to `-1`. Then, the weights that encode these $K$-images are given by:
$$
\begin{equation*}
\mathbf{W} = \frac{1}{K}\cdot\sum_{i=1}^{K}\mathbf{s}_{i}\otimes\mathbf{s}_{i}^{\top}
\end{equation*}
$$
where $\mathbf{s}_{i}$ denotes the state (pixels) of the image we want to memorize, and $\otimes$ denotes the outer product. Thus, the weights are like an average of all of our memories!

* _How big is $K$?_: The maximum theoretical storage limit $K_{\text{max}}$ (the maximum number of possible images that can be stored) of a Hopfield network, using the standard Hebbian learning rule, is approximately $K_{max}\sim{0.138}{N}$, where $N$ is the number of neurons in the network. Thus, the network can reliably store about 14% of its size in patterns before retrieval errors become significant due to interference between stored patterns.

Suppose we've encoded $K$ images and want to retrieve one of them. This seems magical. How does it work? 

### Algorithm: Memory retrieval
Each memory in a Hopfield network is encoded as a _local minimum_ of a global energy function. Thus, during memory retrieval, when we supply a random state vector $\hat{\mathbf{s}}$, we will recover the _closet_ memory encoded in the network to where we start.
The overall energy of the network is given by:
$$
\begin{equation*}
E(\mathbf{s}) = -\frac{1}{2}\cdot\sum_{ij}w_{ij}s_{i}s_{j} - \sum_{i}b_{i}s_{i}
\end{equation*}
$$
where $w_{ij}\in\mathbf{W}$ are the weights of the network, and $b_{i}$ is a bias term. The bias term is used to control the activation of the neurons in the network. The bias term is usually set to zero, but it can be used to control the activation threshold of the neurons in the network.

__Initialize__: Compute the weights $w_{ij}\in\mathbf{W}$ using the Hebbian learning rule, as described above.  Initialize the network with a random state $\mathbf{s}$. Then, use the following algorithm to retrieve a memory:

While __not__ converged:
1. Compute the energy of the network in the current state $\mathbf{s}$ before the update: $E(\mathbf{s})$.
1. Asynchronous update. Choose a random node $i$ and compute a new (potential) state $s_{i}^{\prime}$ using the update rule: $s_{i}^{\prime} \leftarrow \sigma\left(\sum_{j}w_{ij}s_{j}-b_{i}\right)$, where $\sigma$ is the `sign(...)` function and $b_{i}$ is a bias (threshold) parameter.  
2. Compute the energy of the network after the update: $E(\mathbf{s}^{\prime})$. Update the state of the network: $\mathbf{s} \leftarrow \mathbf{s}^{\prime}$.
3. Check for convergence: if the change in the energy of the network is small $(E(\mathbf{s}^{\prime}) - E(\mathbf{s}))^{2} \leq\epsilon$ and we have hit the minimum number of iterations (visited each pixel on average $n$ times), then we have converged.
4. Continue until convergence.

____

## Modern Hopfield Networks
As cool as classical Hopfield networks are, they have limitations. For example, they can only store a limited number of patterns, and are restricted to binary patterns. This is where modern Hopfield networks come in.

The key extension that has given rise to modern Hopfield networks is the reformulation of the energy function. [Krotov and Hopfield (2016)](https://arxiv.org/abs/1606.01164) proposed a new energy function of the form:
$$
\begin{align*}
E(\mathbf{s}) &= -\sum_{i=1}^{K}F(\mathbf{m}_{i}^{\top}\mathbf{s}) \\
\end{align*}
$$
where $F$ is a non-linear function, and $\mathbf{m}_{i}$ is the $i$-th memory, $K$ is the number of memories, and $\mathbf{s}$ is the state of the network. The function $F$ is some non-linear function that maps the inner product of the memory and the state to a scalar value. 

There are many choices for $F$, but, one particulary interesting choice was proposed by [Ramsauer et al (2020)](https://arxiv.org/abs/2008.02217):
$$
\begin{align*}
E(s) &= -\texttt{lse}(\beta,\mathbf{X}^{\top}\mathbf{s}) + \frac{1}{2}\mathbf{s}^{\top}\mathbf{s} + \frac{1}{\beta}\log(K)+ \frac{1}{2}M^{2} \\
\end{align*}
$$
where $\mathbf{X}\in\mathbb{R}^{N\times{K}}$ is the matrix of memories, i.e., each memory $\mathbf{m}_{1},\dots,\mathbf{m}_{K}$ consisting of $N$ features is a column of the matrix, $\mathbf{s}$ is the current state of the network and $\texttt{lse}(\cdot)$ is the log-sum-exp function:
$$
\begin{align*}
\texttt{lse}(\beta,\mathbf{z}) &= \frac{1}{\beta}\log\left(\sum_{i=1}^{K}\exp(\beta\,\mathbf{z}_{i})\right) \\
\end{align*}
$$
and $\beta$ is an inverse temperature parameter that controls the sharpness of the distribution. Finally, $M$ is the largest norm of all the memories, i.e., $M = \max_{i=1,\dots,K}\|\mathbf{m}_{i}\|$.

### Algorithm: Memory retrieval
The user provides a set of memory vectors $\mathbf{X} = \left\{\mathbf{m}_{1}, \mathbf{m}_{2}, \ldots, \mathbf{m}_{K}\right\}$, where $\mathbf{m}_{i} \in \mathbb{R}^{N}$ is a memory vector of size $N$ and $K$ is the number of memory vectors. Further, the user provides an initial _partial memory_ $\mathbf{s}_{\circ} \in \mathbb{R}^{N}$, which is a vector of size $N$ and specifies the _inverse temperature_ $\beta$ of the system.

__Initialize__ the network with the memory vectors $\mathbf{X}$, and the inverse temperature $\beta$. Set current state to the initial state $\mathbf{s} \gets \mathbf{s}_{\circ}$

Until convergence __do__:
   1. Compute the _current_ probability vector defined as $\mathbf{p} = \texttt{softmax}(\beta\cdot\mathbf{X}^{\top}\mathbf{s})$ where $\mathbf{s}$ is the _current_ state vector, and $\mathbf{X}^{\top}$ is the transpose of the memory matrix.
   2. Compute the _next_ state vector $\mathbf{s}^{\prime} = \mathbf{X}\mathbf{p}$ using the current probability vector $\mathbf{p}$ and the memory matrix $\mathbf{X}$. This step computes a weighted sum of the memory vectors based on the probabilities.
   3. Check for convergence. If $\mathbf{s}^{\prime}$ is _close_ to $\mathbf{s}$ or we run out of iterations, then __stop__. For example, $\lVert \mathbf{s}^{\prime} - \mathbf{s}\rVert_{2}^{2} \leq \epsilon$ for some small $\epsilon > 0$.
   4. Otherwise, update the state $\mathbf{s} \gets\mathbf{s}^{\prime}$, and __go back to__ step 1.

   
### Why is this so interesting?
The energy function is a generalization of the classical Hopfield network energy function, and it allows for the storage of continuous patterns, thus $\mathbf{m}_{i}\in\mathbb{R}^{N}$, the number of memories that can be stored is exponential in the number of neurons, and the memories can be correlated.

___